# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import DataType, StringType

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

# Transformations

- Trim text columns
- Remove NAS from the start of CID to match up with customer id in crm_cust_info
- Birthday cannot be greater than current date so set these to NULL/None
- Normalize gender column
- Rename colum headers


### Trim text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Cleaning customer ID column

In [0]:
df = df.withColumn(
    "CID",
    F.when(col("CID").startswith("NAS"),
           F.substring(col("CID"), 4, F.length(col("CID"))))
        .otherwise(col("CID"))
)

### Clean birthday date column

In [0]:
df = df.withColumn(
    "BDATE",
    F.when(col("BDATE") > F.current_date(), None)
     .otherwise(col("BDATE"))
)

### Normalize gender column

In [0]:
df = df.withColumn(
    "GEN",
    F.when(col("GEN").isin("F", "Female"), "Female")
     .when(col("GEN").isin("M", "Male"), "Male")
     .otherwise("N/A")
)

### Renaming column headers

In [0]:
RENAME_MAPPING = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}

for old_name, new_name in RENAME_MAPPING.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to silver table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customers")

### Sanity check silver table

In [0]:
%sql 

SELECT * FROM workspace.silver.erp_customers LIMIT 10